In [1]:
import numpy as np
# ----------------------------
# EOS-80 freezing point (ITS-90)
# t_f(°C, ITS-90) = (a0*S + a1*S*sqrt(S) + a2*S^2 + b*p) × 0.99976
# a0..b are UNESCO / Millero coefficients (originally on IPTS-68).
# Multiply by 0.99976 to convert to ITS-90.  (1/1.00024 ≈ 0.99976)
# Refs: UNESCO 1983; Millero & Leung (1976); CSIRO sw_fp.m implementation.
# ----------------------------
from seawater import fp

# ----------------------------
# Practical Salinity from conductivity (PSS-78)
# Try python-seawater (EOS-80) first; fall back to TEOS-10 gsw.SP_from_C.
# Both implement the PSS-78 practical-salinity algorithm from UNESCO (1983).
# ----------------------------
from gsw import SP_from_C

# ----------------------------
# Monte-Carlo uncertainty propagation for SP and EOS-80 freezing point
# ----------------------------
def propagate_uncertainty_eos80(
    C_Sm, T90_C, p_dbar,
    uC_Sm, uT_C, uP_dbar,
    n_samples=20000, seed=0, clip_C_positive=True, return_fd_check=True
):
    """
    Inputs may be scalars or arrays (they will be broadcast to common shape).
    Instrument 1-sigma uncertainties: uC (mS/cm), uT (°C ITS-90), uP (dbar).
    Returns dict with central values and 1-sigma uncertainties from Monte Carlo.
    """
    rng = np.random.default_rng(seed)

    # Broadcast to common shape
    C0 = np.asarray(C_Sm, float)
    T0 = np.asarray(T90_C, float)
    P0 = np.asarray(p_dbar, float)
    shape = np.broadcast_shapes(C0.shape, T0.shape, P0.shape)
    C0 = np.broadcast_to(C0, shape)
    T0 = np.broadcast_to(T0, shape)
    P0 = np.broadcast_to(P0, shape)

    # Draw samples
    C_s = rng.normal(C0, uC_Sm, size=(n_samples,) + shape)
    T_s = rng.normal(T0, uT_C,     size=(n_samples,) + shape)
    P_s = rng.normal(P0, uP_dbar,  size=(n_samples,) + shape)

    if clip_C_positive:
        C_s = np.maximum(C_s, np.finfo(float).tiny)  # avoid non-positive C on log/ratio ops

    # Convert C to units mS/cm for GSW toolbox
    C_s_mScm = 10 * C_s

    # Evaluate SP and freezing point for each draw
    # (loop by chunk to keep memory in check if your arrays are large)
    SP_s = np.empty_like(C_s)
    for k in range(n_samples):
        SP_s[k] = SP_from_C(C_s_mScm[k], T_s[k], P_s[k]) # replaced with original GSW package

    Tf_s = fp(SP_s, P_s) # replaced with original EOS80 package

    # Central values at nominal inputs
    SP0 = SP_from_C(10*C0, T0, P0)
    Tf0 = fp(SP0, P0)

    # Monte-Carlo 1-sigma
    u_SP = SP_s.std(axis=0, ddof=1)
    u_Tf = Tf_s.std(axis=0, ddof=1)

    out = {
        "SP": SP0,        # Practical Salinity (PSS-78)
        "Tf": Tf0,        # Freezing point (°C, ITS-90)
        "u_SP": u_SP,     # 1-sigma uncertainty in SP (PSS-78 units)
        "u_Tf": u_Tf,     # 1-sigma uncertainty in Tf (°C)
        "n_samples": n_samples
    }

    return out

/tmp/ipykernel_62874/4179690754.py:9: UserWarning: The seawater library is deprecated! Please use gsw instead.
  from seawater import fp


In [2]:
# Example nominal values:
C_Sm = 2.713       # conductivity [S/m] (roughly expected at the measurement site;
                # TO-DO: find value from upper ~40 m CTD / mixed; Fig. 7a)
T90_C   = -1.94      # in-situ temperature [°C, ITS-90] (from upper ~40 m CTD / mixed; Fig. 7a)

# evaluate the pressure for the instrument depth
from gsw import p_from_z
# camp 2021 coordinates
lon = 166.23333333333332
lat = -77.86666666666666
p_dbar  = p_from_z(-100, lat)     # sea pressure [dbar]

# Instrument 1-sigma uncertainties:
delta_t = 4 # months; calibration date CT 30 June 2021
uC_Sm = 0.0005 + delta_t * 0.0003     # S/m (SBE 4C data sheet)
uT_C    = 0.0001 + delta_t * (1/6) * 0.0001   # deg C (SBE 3plus data sheet)
uP_dbar = 0.       # 0.5 dbar # assume insignificant errors in pressure measurements (SBE19plus data sheet)

res = propagate_uncertainty_eos80(
    C_Sm, T90_C, p_dbar,
    uC_Sm, uT_C, uP_dbar,
    n_samples=5000, seed=123
)

print("SP =", res["SP"], "±", res["u_SP"], "(Monte Carlo 1σ)")
print("Tf =", res["Tf"], "±", res["u_Tf"], "°C (Monte Carlo 1σ)")
print("relative error SP: ", abs(res["u_SP"]/res["SP"]))
print("relative error Tf: ", abs(res["u_Tf"]/res["Tf"]))
print("relative error T90: ", abs(uT_C/T90_C))

SP = 34.5767451021336 ± 0.023906050669719296 (Monte Carlo 1σ)
Tf = -1.9736677645170284 ± 0.001369850232959119 °C (Monte Carlo 1σ)
relative error SP:  0.0006913910085840943
relative error Tf:  0.000694063234748292
relative error T90:  8.591065292096222e-05



# Supplementary material: errors for in-situ temperature, practical salinity, in-situ supercooling and potential density

## a) In-situ temperature

### SBE 3plus
Manufacturer‑stated temperature uncertainty:
$$
\sigma_{1,T} = 0.0001 + \Delta t \cdot \frac{1}{6} \cdot 0.0001
$$
where $\Delta_t$ was 4 months and temperature in °C (Sea-Bird Electronics, Inc. 2026b).

### SBE 19plus
Manufacturer‑stated temperature uncertainty:
$$
\sigma_{1,T} = 0.005 + \Delta t \cdot 0.0002
$$
°C (Sea-Bird Electronics, Inc. 2026a).

---

## b) Practical salinity

Uncertainties in practical salinity were obtained by perturbing temperature, conductivity, and pressure using normally distributed errors (EOS80).

### Conductivity
**SBE 4C:**
$$
\sigma_{1,C} = 0.0005 + \Delta t \cdot 0.0003
$$
in S/m (Sea-Bird Electronics, Inc. 2026c).

**SBE 19plus:**
$$
\sigma_{1,C} = 0.0003 + \Delta t \cdot 0.0003
$$

### Pressure (SBE 19plus)
Pressure uncertainty was computed from the manufacturer‑specified initial accuracy and drift terms:

$$
\sigma_{1,P}
= 0.1 \cdot 10^{-2} P
+ 0.02 \cdot 10^{-2} P
+ \frac{1}{12} \cdot 0.02 \cdot 10^{-2} P \cdot \Delta t
$$

---

### Simulation (n = 20,000)

Temperature, conductivity and pressure were sampled as:
$$
T_s \sim \mathcal{N}(T, \sigma_{1,T})
$$
$$
C_s \sim \mathcal{N}(C, \sigma_{1,C})
$$
$$
P_s \sim \mathcal{N}(P, \sigma_{1,P})
$$

Practical salinity was computed for each sample:
$$
SP_s = SP(C_s, T_s, P_s)
$$

Uncertainty:
$$
\sigma_{1,SP} = sd(SP)
$$

---

## c) In-situ supercooling

Supercooling uncertainty was obtained by combining temperature and freezing‑point uncertainties in quadrature (Taylor 1996):
$$
\sigma_{1,SC} = \sqrt{\sigma_{1,T}^2 + \sigma_{1,T_f}^2}
$$

### Freezing point temperature
Freezing point temperature was evaluated from perturbed salinity and pressure:
$$
T_{f,s} = FP(SP_s, P_s)
$$

Uncertainty:
$$
\sigma_{1,T_f} = sd(T_{f,s})
$$

---

## d) Potential density

Potential density was simulated using **pden** (EOS80):
$$
\rho_s = pden(SP_s, T_s / 0.99976, P_s)
$$

Uncertainty:
$$
\sigma_{1,\rho} = sd(\rho_s)
$$
``
